# Phase 9 — QLoRA fine-tuning of Qwen3-8B for Text-to-SQL

Runs on a **free Colab T4**. Nothing here touches `dataset/test/test.jsonl`,
and no benchmark is run — evaluation is Phase 10.

**Before anything else:** `Runtime -> Change runtime type -> T4 GPU`.

| | |
|---|---|
| Base model | `Qwen/Qwen3-8B` @ `b968826d9c46` |
| Method | QLoRA — 4-bit NF4 base, LoRA r=16 adapters |
| Sequence length | 2048 |
| Training data | 2,133 examples (`dataset/sft/train.jsonl`) |
| Validation | 459 examples |
| Expected VRAM | ~9-11 GB of 15 GB |
| Expected time | ~1.5 h/epoch, ~3 h for 2 epochs |

> Free Colab sessions can be reclaimed at any time. Checkpoints are written
> every 50 steps; **save them to Drive** (cell 3) so an interrupted run
> resumes instead of restarting.


## 1. GPU check

Stop here if this does not show a GPU. The training script refuses to run
on CPU by design — an 8B model would take months.


In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU'
p = torch.cuda.get_device_properties(0)
print(f'{p.name} | {p.total_memory/1024**3:.1f} GB | compute {p.major}.{p.minor}')
print('bf16 supported     :', p.major >= 8)
print('FlashAttention-2 ok:', p.major >= 8)
print('-> T4 (7.5) uses fp16 + SDPA automatically')


## 2. Install dependencies

Colab's preinstalled CUDA build of torch is reused deliberately —
reinstalling it wastes several GB and many minutes.

**Restart the session afterwards** (`Runtime -> Restart session`), then
continue from cell 3. Skipping the restart is the most common cause of
`bitsandbytes` failing to find CUDA.


In [ ]:
# Loose upper bounds on purpose: Colab ships Python 3.13 / torch 2.11 /
# CUDA 12.8, and older pinned bitsandbytes fails on `triton.ops`
# (removed in Triton 3.0) with no binary for CUDA 12.8.
!pip install -q -U bitsandbytes transformers peft accelerate datasets trl sentencepiece

import importlib.metadata as md
for pkg in ['torch','transformers','peft','bitsandbytes','accelerate','datasets','trl']:
    try:
        print(f'{pkg:<15}{md.version(pkg)}')
    except Exception:
        print(f'{pkg:<15}NOT INSTALLED')

print()
print('installed - now Runtime -> Restart session, then run cell 3 onward')


## 3. Mount Drive and get the code + data

Drive matters more than convenience here: it is what makes an interrupted
free-tier session resumable rather than wasted.

Pick **one** of the three options below.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/text2sql-qlora'
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
print('working directory:', os.getcwd())


### Option A — clone from GitHub (cleanest, if the repo is pushed)


In [ ]:
# !git clone https://github.com/<you>/<repo>.git repo
# %cd repo
# !ls dataset/sft/


### Option B — upload the two files by hand

You only need `train.jsonl` (14 MB) and `validation.jsonl` (3 MB) plus the
training script — not the whole project.


In [ ]:
import os, shutil
os.makedirs('dataset/sft', exist_ok=True)
os.makedirs('training/qlora', exist_ok=True)

from google.colab import files
print('Select: train.jsonl, validation.jsonl, train_qlora.py')
uploaded = files.upload()

for name in uploaded:
    dest = 'training/qlora/' if name.endswith('.py') else 'dataset/sft/'
    shutil.move(name, dest + name)
print(os.listdir('dataset/sft'), os.listdir('training/qlora'))


### Option C — copy from a Drive folder you uploaded earlier


In [ ]:
# !mkdir -p dataset/sft training/qlora
# !cp /content/drive/MyDrive/text2sql-data/*.jsonl dataset/sft/
# !cp /content/drive/MyDrive/text2sql-data/train_qlora.py training/qlora/


## 4. Verify the dataset arrived intact

The content hashes must match the values frozen in Phase 7. A mismatch
means the training data is not the audited data, and any later comparison
against the 10.82 % baseline would be meaningless.


In [ ]:
import json, hashlib

EXPECTED = {'train': '7804d4386c9b1004', 'validation': '43a469080798f5aa'}

for split, expected in EXPECTED.items():
    rows = [json.loads(l) for l in open(f'dataset/sft/{split}.jsonl', encoding='utf-8')]
    h = hashlib.sha256()
    for r in rows:
        for m in r['messages']:
            h.update(m['role'].encode()); h.update(bytes([0]))
            h.update(m['content'].encode()); h.update(bytes([1]))
    got = h.hexdigest()[:16]
    print(f'{split:<11} {len(rows):>5} records  {got}  '
          f"{'MATCH' if got == expected else 'MISMATCH !!'}")
    assert got == expected, f'{split} content hash differs from the frozen dataset'

print()
print('dataset verified against the Phase 7 freeze')


## 5. Smoke test — 8 steps

Proves the whole pipeline works (download, quantise, mask, step, save)
before committing three hours to it. Takes ~10 minutes, most of which is
the 16 GB model download that the real run then reuses from cache.


In [ ]:
!python training/qlora/train_qlora.py --smoke --output-dir outputs/smoke


## 6. Full training run

~3 hours for 2 epochs on a T4. Checkpoints land in Drive every 50 steps.

**If the session dies, re-run cells 1, 3 and this one with `--resume`.**
Nothing is lost.


In [ ]:
!python training/qlora/train_qlora.py \
    --train-file dataset/sft/train.jsonl \
    --val-file dataset/sft/validation.jsonl \
    --output-dir outputs/qwen3-8b-text2sql-qlora \
    --epochs 2


### Resume after a disconnect


In [ ]:
# !python training/qlora/train_qlora.py --resume \
#     --output-dir outputs/qwen3-8b-text2sql-qlora --epochs 2


## 7. Inspect what was produced


In [ ]:
import json, os
OUT = 'outputs/qwen3-8b-text2sql-qlora'

for root, _, files in os.walk(OUT):
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f'{os.path.getsize(p)/1e6:8.1f} MB  {p}')

m = json.load(open(f'{OUT}/training_metrics.json'))
print()
print('runtime    :', m['train_runtime_h'], 'h')
print('final loss :', m['train_loss'])
print('final eval :', m.get('final_eval'))


## 8. Download the adapter

The adapter is ~80-160 MB — the 16 GB base model is *not* included, and
does not need to be: Phase 10 loads `Qwen/Qwen3-8B` and applies this on top.

If `WORK` is inside Drive it is already saved; this cell is only needed for
a local copy.


In [ ]:
!cd outputs/qwen3-8b-text2sql-qlora && zip -qr /content/adapter.zip \
    final_adapter training_config.json training_metrics.json

from google.colab import files
files.download('/content/adapter.zip')
print('place the contents in models/finetuned/ in the project repo')


---

## What this notebook deliberately does not do

- **No benchmark run.** The 453-example test set is Phase 10's job, and it
  must run through the same harness that produced the frozen baseline.
- **No dataset changes.** `dataset/sft/*` is read-only here, and its hashes
  are asserted in cell 4.
- **No CPU fallback.** The script exits with an explanation instead.

### The one detail worth understanding

Each example is ~1,473 prompt tokens and ~45 completion tokens, and the
prompt is ~97 % database schema identical across every example.

The script masks the prompt out of the loss and trains on the SQL only.
Without that masking, ~97 % of the gradient would go into memorising a
schema the model is *handed* at inference — the loss curve would look
excellent while the ability you actually care about barely moved.
